In [1]:
import pandas as pd
import numpy as np


raw_data = pd.read_csv("../raw_data/nfl_odds_2007-2022.csv", parse_dates=["date"])
raw_data.head()

,date,away_team,home_team,away_score,home_score,away_ml,home_ml,away_spread,home_spread,close_total,season,home_win
0,2007-09-06,NewOrleans,Indianapolis,10,41,200,-240,0.0,5.5,52.5,2007-08,1.0
1,2007-09-09,KansasCity,HoustonTexans,3,20,170,-200,0.0,3.0,37.5,2007-08,1.0
2,2007-09-09,Denver,Buffalo,15,14,-170,150,3.0,0.0,38.5,2007-08,0.0
3,2007-09-09,Pittsburgh,Cleveland,34,7,-235,195,5.0,0.0,36.0,2007-08,0.0
4,2007-09-09,Tennessee,Jacksonville,13,10,290,-350,0.0,7.5,39.5,2007-08,0.0


### No-vig (Vig removal)
Every line (moneyline, closed spread, closed total) all have some "juice" in it which is a percentage that the house makes. Removing the vig is needed to convert the odds to implied probabilities. Not removing vig would lead to probabilities >100% due to the built in edge. 

In [2]:
def ml_vig(fav_odds, udog_odds):
    """Removes the vig from moneyline.

    :param fav_odds: Favorite odds (usually negative).
    :param udog_odds: Underdog odds (usually positive).
    return : Implied probability for the game
    """
    probabilities = {
        "fav_prob" : '',
        "udog_prob" : '',
        "vig_prob" : ''
    } 

    #The individual probabilities for favorites and underdogs to win the game
    fav_prob = abs(fav_odds) / (abs(fav_odds) + 100)
    udog_prob = 100 / (udog_odds + 100)

    total_prob = fav_prob + udog_prob
    probabilities['vig_prob'] = round(total_prob - 1, 2)

    #Removing the Vig (normalized prob)
    probabilities["fav_prob"] = round(fav_prob / total_prob , 2)
    probabilities["udog_prob"] = round(udog_prob / total_prob, 2) 

    #Probabilities in dictionary will not add up to 1 because the favorite and underdog
    #probaility are already normalized and the vig was from before they were normalized
    return probabilities

for i in range(6):
    print(ml_vig(*sorted((raw_data['away_ml'][i], raw_data['home_ml'][i]))))

{'fav_prob': 0.68, 'udog_prob': 0.32, 'vig_prob': 0.04}
{'fav_prob': 0.64, 'udog_prob': 0.36, 'vig_prob': 0.04}
{'fav_prob': 0.61, 'udog_prob': 0.39, 'vig_prob': 0.03}
{'fav_prob': 0.67, 'udog_prob': 0.33, 'vig_prob': 0.04}
{'fav_prob': 0.75, 'udog_prob': 0.25, 'vig_prob': 0.03}
{'fav_prob': 0.53, 'udog_prob': 0.47, 'vig_prob': 0.04}


In [3]:
import statsmodels.api as sm
def spread_prob():
    """
    Determines probability of home/away team win based on closed spread
    using logistic regression.

    Uses both away_spread and home_spread as predictors. Only one of the
    two is ever nonzero for a given game (whichever team is favored), so
    together they tell the model both the size of the spread and which
    team it favors -- a single unsigned spread column can't do that.

    Adds two columns: home_spread_prob (home team win probability) and
    away_spread_prob (away team win probability, i.e. 1 - home_spread_prob).
    """
    #fitting the data
    X = sm.add_constant(raw_data[["away_spread", "home_spread"]])
    model = sm.Logit(raw_data["home_win"], X).fit()

    home_win_prob = model.predict(X)
    raw_data["home_spread_prob"] = round(home_win_prob, 2)
    raw_data["away_spread_prob"] = round(1 - home_win_prob, 2)

    
spread_prob()
raw_data.head(10)

Optimization terminated successfully.
         Current function value: 0.608198
         Iterations 5


,date,away_team,home_team,away_score,home_score,away_ml,home_ml,away_spread,home_spread,close_total,season,home_win,home_spread_prob,away_spread_prob
0,2007-09-06,NewOrleans,Indianapolis,10,41,200,-240,0.0,5.5,52.5,2007-08,1.0,0.68,0.32
1,2007-09-09,KansasCity,HoustonTexans,3,20,170,-200,0.0,3.0,37.5,2007-08,1.0,0.58,0.42
2,2007-09-09,Denver,Buffalo,15,14,-170,150,3.0,0.0,38.5,2007-08,0.0,0.38,0.62
3,2007-09-09,Pittsburgh,Cleveland,34,7,-235,195,5.0,0.0,36.0,2007-08,0.0,0.33,0.67
4,2007-09-09,Tennessee,Jacksonville,13,10,290,-350,0.0,7.5,39.5,2007-08,0.0,0.74,0.26
5,2007-09-09,Carolina,St.Louis,27,13,105,-125,0.0,2.5,43.0,2007-08,0.0,0.57,0.43
6,2007-09-09,Philadelphia,GreenBay,13,16,-190,160,3.0,0.0,41.5,2007-08,1.0,0.38,0.62
7,2007-09-09,Atlanta,Minnesota,3,24,150,-170,0.0,3.0,34.0,2007-08,1.0,0.58,0.42
8,2007-09-09,Miami,Washington,13,16,155,-175,0.0,3.0,34.0,2007-08,1.0,0.58,0.42
9,2007-09-09,NewEngland,NYJets,38,14,-260,220,6.0,0.0,41.5,2007-08,0.0,0.30,0.70
